# Product-invariant EV-battery disassembly ranking: MLP vs GNN under LOPO

This notebook is the **first neural feasibility experiment** after the non-neural baseline audit.

The task is deliberately narrow:

> given the current reference-sequence operation and the remaining operation inventory of a battery, rank the **reference next operation**.

The held-out test unit is an **entire battery pack**. No battery identity, product name, raw part ID, fixation ID, or operation ID is provided to either model.

## Models

### MLP
Uses only product-invariant operation attributes:

- skill;
- fixation/part-removal kind;
- fixation class;
- source/target part classes;
- removed-part class;
- normalized local degrees;
- current-operation context;
- sequence progress.

### GNN
Uses the same attributes **plus a two-layer message-passing encoding of the battery part/fixation graph**.

This isolates whether relational topology adds value beyond local categorical attributes.

## Evaluation

For each of 10 leave-one-battery-pack-out folds:

- train on 9 batteries;
- test on the completely unseen 10th battery;
- repeat neural training with 3 fixed random seeds;
- rank all remaining reference operations at every decision position.

Metrics:

- `Hits@1`
- `Hits@3`
- `MRR`

Ties are handled by **expected rank under random tie-breaking**, so the MLP is not accidentally rewarded or penalized by operation-ID/order tie breaking.

We also report performance separately when the held-out next-operation **full relational signature** was:

- seen in the nine training batteries;
- unseen in the nine training batteries.

## Frozen continuation gate

Before any neural results are inspected, continue to a more sophisticated graph model only if **all three** conditions hold:

1. mean LOPO `Hits@1(GNN) - Hits@1(MLP) >= 0.05`;
2. GNN has higher fold-mean Hits@1 on at least **7 of 10** held-out batteries;
3. mean GNN MRR is higher than mean MLP MRR.

This is a feasibility gate, not a statistical-significance test.

### Important scope

The KIT dataset contains one reference disassembly sequence per product and does not enumerate all equivalent valid alternatives. Therefore this notebook evaluates **reference-sequence ranking**, not physical-validity classification.

In [ ]:
# 1. Imports and frozen configuration

from pathlib import Path
from collections import Counter, defaultdict
import tarfile, json, math, random, time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

SEEDS = [17, 29, 43]

EPOCHS = 30
NEGATIVES_PER_STATE = 8
LR = 2e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10

EMBED_DIM = 12
HIDDEN_DIM = 64
GNN_DIM = 32

BASE_SEED = 20261003

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("Neural fits:", 10 * 2 * len(SEEDS))

In [ ]:
# 2. Find the KIT dataset automatically.
#
# The cell first searches for already-extracted CSVs anywhere under /content.
# If not found, it searches for the uploaded KIT .tar archive and extracts it.

root = Path("/content")

catalogues = list(root.rglob("battery_catalogue.csv"))

if catalogues:
    DATA = catalogues[0].parent
else:
    archives = list(root.rglob("10.35097-emz24pksshndq468.tar"))
    if not archives:
        archives = list(root.rglob("10.35097-*.tar"))

    assert archives, (
        "KIT dataset not found. Upload 10.35097-emz24pksshndq468.tar "
        "to the Colab Files panel, then rerun this cell."
    )

    TAR_PATH = archives[0]
    print("Using archive:", TAR_PATH)

    work = root / "battery_disassembly_dataset_extracted"
    work.mkdir(parents=True, exist_ok=True)

    with tarfile.open(TAR_PATH, "r") as tf:
        tf.extractall(work)

    catalogues = list(work.rglob("battery_catalogue.csv"))
    assert catalogues, "battery_catalogue.csv not found after extraction."
    DATA = catalogues[0].parent

print("Dataset directory:", DATA)

In [ ]:
# 3. Load and validate raw data.

bat = pd.read_csv(DATA / "battery_catalogue.csv")
parts = pd.read_csv(DATA / "parts.csv")
fix = pd.read_csv(DATA / "fixations.csv")
ops = pd.read_csv(DATA / "operations.csv")

print("Batteries:", len(bat))
print("Parts:", len(parts))
print("Fixations:", len(fix))
print("Operations:", len(ops))

display(bat[["id", "name"]])

In [ ]:
# 4. Attach each operation to a battery and reconstruct the 10 reference chains.

battery_names = bat.set_index("id")["name"].to_dict()

fix_battery = fix.set_index("id")["battery_id"].to_dict()
part_battery = parts.set_index("id")["battery_id"].to_dict()
part_class_raw = parts.set_index("id")["class"].to_dict()

fix_info_raw = (
    fix.set_index("id")[["class", "fromPartID", "toPartID"]]
    .to_dict("index")
)

def operation_battery(row):
    if pd.notna(row.fixation_id):
        return int(fix_battery[int(row.fixation_id)])
    return int(part_battery[int(row.part_id)])

ops["battery_id"] = ops.apply(operation_battery, axis=1)
ops_by_id = ops.set_index("id")

chains = {}

for b in sorted(ops["battery_id"].unique()):
    sub = ops[ops["battery_id"] == b]
    ids = set(sub["id"].astype(int))
    next_ids = set(sub["next_op_id"].dropna().astype(int))

    starts = list(ids - next_ids)
    terminals = sub[sub["next_op_id"].isna()]["id"].astype(int).tolist()

    assert len(starts) == 1
    assert len(terminals) == 1

    seq = []
    seen = set()
    cur = int(starts[0])

    while cur in ids and cur not in seen:
        seen.add(cur)
        seq.append(cur)

        nxt = ops_by_id.loc[cur, "next_op_id"]
        if pd.isna(nxt):
            break
        cur = int(nxt)

    assert len(seq) == len(ids), f"Broken chain for battery {b}"
    chains[int(b)] = seq

print({
    battery_names[b]: len(seq)
    for b, seq in chains.items()
})

In [ ]:
# 5. Raw full relational signatures.
#
# These are used only for diagnostics (seen vs unseen motif/signature).
# They are NOT output classes for the neural ranking models.

def raw_full_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)

    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        src_cls = str(part_class_raw[int(fi["fromPartID"])])
        dst_cls = str(part_class_raw[int(fi["toPartID"])])
        return (
            f"{skill}|fix:{fi['class']}|"
            f"{src_cls}->{dst_cls}"
        )

    return (
        f"{skill}|part:"
        f"{part_class_raw[int(r.part_id)]}"
    )

raw_signatures = {
    b: [raw_full_signature(op_id) for op_id in seq]
    for b, seq in chains.items()
}

In [ ]:
# 6. Fold-specific vocabularies.
#
# Vocabularies are learned ONLY from the 9 training batteries.
# Category 0 is <UNK>, so truly unseen test categories do not leak into training.

def make_vocab(values):
    vals = sorted({
        str(x)
        for x in values
        if pd.notna(x)
    })
    return {
        "<UNK>": 0,
        **{v: i + 1 for i, v in enumerate(vals)}
    }

def lookup(vocab, value):
    if pd.isna(value):
        return 0
    return vocab.get(str(value), 0)

def build_fold_vocab(train_batteries):
    return {
        "part": make_vocab(
            parts[
                parts["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "fix": make_vocab(
            fix[
                fix["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "skill": make_vocab(
            ops[
                ops["battery_id"].isin(train_batteries)
            ]["skill_id"]
        ),
    }

In [ ]:
# 7. Convert one battery into product-invariant graph + operation features.

def build_battery_pack(battery_id, vocab):
    psub = parts[
        parts["battery_id"] == battery_id
    ].copy()

    fsub = fix[
        fix["battery_id"] == battery_id
    ].copy()

    part_id_to_local = {
        int(pid): i
        for i, pid in enumerate(psub["id"].astype(int))
    }

    n_nodes = len(psub)

    degree = np.zeros(n_nodes, dtype=np.float32)

    edge_src = []
    edge_dst = []
    edge_type = []

    for _, r in fsub.iterrows():
        a = part_id_to_local[int(r.fromPartID)]
        b = part_id_to_local[int(r.toPartID)]

        degree[a] += 1
        degree[b] += 1

        # Undirected message-passing view of physical fixation relation.
        edge_src += [a, b]
        edge_dst += [b, a]

        et = lookup(vocab["fix"], r["class"])
        edge_type += [et, et]

    degree_norm = degree / max(1.0, float(degree.max()))

    node_part_class = np.array(
        [
            lookup(vocab["part"], cls)
            for cls in psub["class"]
        ],
        dtype=np.int64,
    )

    edge_index = np.array(
        [edge_src, edge_dst],
        dtype=np.int64,
    )

    edge_type = np.array(
        edge_type,
        dtype=np.int64,
    )

    seq = [int(x) for x in chains[battery_id]]

    categorical = []
    graph_nodes = []
    numeric = []
    signatures = []

    for op_id in seq:
        r = ops_by_id.loc[op_id]

        skill_idx = lookup(
            vocab["skill"],
            r.skill_id,
        )

        if pd.notna(r.fixation_id):
            fi = fix_info_raw[int(r.fixation_id)]

            src_pid = int(fi["fromPartID"])
            dst_pid = int(fi["toPartID"])

            src_node = part_id_to_local[src_pid]
            dst_node = part_id_to_local[dst_pid]

            categorical.append([
                skill_idx,
                0,  # operation kind: fixation
                lookup(vocab["fix"], fi["class"]),
                lookup(vocab["part"], part_class_raw[src_pid]),
                lookup(vocab["part"], part_class_raw[dst_pid]),
                0,  # removed-part class is not applicable
            ])

            graph_nodes.append([
                src_node,
                dst_node,
                -1,
            ])

            numeric.append([
                degree_norm[src_node],
                degree_norm[dst_node],
                0.0,
            ])

        else:
            pid = int(r.part_id)
            node = part_id_to_local[pid]

            categorical.append([
                skill_idx,
                1,  # operation kind: part removal
                0,
                0,
                0,
                lookup(vocab["part"], part_class_raw[pid]),
            ])

            graph_nodes.append([
                -1,
                -1,
                node,
            ])

            numeric.append([
                0.0,
                0.0,
                degree_norm[node],
            ])

        signatures.append(raw_full_signature(op_id))

    return {
        "battery_id": int(battery_id),
        "battery_name": battery_names[int(battery_id)],
        "node_part_class": node_part_class,
        "node_degree": degree_norm.astype(np.float32),
        "edge_index": edge_index,
        "edge_type": edge_type,
        "op_categorical": np.asarray(categorical, dtype=np.int64),
        "op_graph_nodes": np.asarray(graph_nodes, dtype=np.int64),
        "op_numeric": np.asarray(numeric, dtype=np.float32),
        "full_signatures": signatures,
        "n_ops": len(seq),
    }

In [ ]:
# 8. Neural models.

class BaseOperationRanker(nn.Module):
    """
    Product-invariant MLP operation ranker.

    It never receives battery/product IDs or raw entity IDs.
    """

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.skill_emb = nn.Embedding(n_skill, emb_dim)
        self.kind_emb = nn.Embedding(2, 4)
        self.fix_emb = nn.Embedding(n_fix, emb_dim)
        self.part_emb = nn.Embedding(n_part, emb_dim)

        op_input_dim = (
            emb_dim      # skill
            + 4          # operation kind
            + emb_dim    # fixation class
            + emb_dim    # source part class
            + emb_dim    # target part class
            + emb_dim    # removed part class
            + 3          # normalized local degrees
        )

        self.op_mlp = nn.Sequential(
            nn.Linear(op_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # current, candidate, abs difference, product, progress
        self.score_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def base_operation_encoding(self, pack):
        cats = torch.as_tensor(
            pack["op_categorical"],
            dtype=torch.long,
            device=DEVICE,
        )

        nums = torch.as_tensor(
            pack["op_numeric"],
            dtype=torch.float32,
            device=DEVICE,
        )

        skill = cats[:, 0]
        kind = cats[:, 1]
        fix_class = cats[:, 2]
        src_class = cats[:, 3]
        dst_class = cats[:, 4]
        part_class = cats[:, 5]

        x = torch.cat([
            self.skill_emb(skill),
            self.kind_emb(kind),
            self.fix_emb(fix_class),
            self.part_emb(src_class),
            self.part_emb(dst_class),
            self.part_emb(part_class),
            nums,
        ], dim=-1)

        return self.op_mlp(x)

    def encode_operations(self, pack):
        return self.base_operation_encoding(pack)

    def score_pairs(self, current_repr, candidate_repr, progress):
        x = torch.cat([
            current_repr,
            candidate_repr,
            torch.abs(current_repr - candidate_repr),
            current_repr * candidate_repr,
            progress,
        ], dim=-1)

        return self.score_mlp(x).squeeze(-1)


class GraphEncoder(nn.Module):
    """
    Two-layer product-invariant message-passing network.
    """

    def __init__(
        self,
        n_part,
        n_fix,
        part_emb_dim=EMBED_DIM,
        edge_emb_dim=8,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.part_emb = nn.Embedding(n_part, part_emb_dim)
        self.edge_emb = nn.Embedding(n_fix, edge_emb_dim)

        self.input_proj = nn.Linear(
            part_emb_dim + 1,
            graph_dim,
        )

        self.message_layers = nn.ModuleList([
            nn.Linear(
                graph_dim + edge_emb_dim,
                graph_dim,
            )
            for _ in range(2)
        ])

        self.self_layers = nn.ModuleList([
            nn.Linear(graph_dim, graph_dim)
            for _ in range(2)
        ])

        self.dropout = dropout

    def forward(self, pack):
        node_class = torch.as_tensor(
            pack["node_part_class"],
            dtype=torch.long,
            device=DEVICE,
        )

        degree = torch.as_tensor(
            pack["node_degree"],
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(-1)

        h = F.relu(
            self.input_proj(
                torch.cat([
                    self.part_emb(node_class),
                    degree,
                ], dim=-1)
            )
        )

        edge_index = torch.as_tensor(
            pack["edge_index"],
            dtype=torch.long,
            device=DEVICE,
        )

        edge_type = torch.as_tensor(
            pack["edge_type"],
            dtype=torch.long,
            device=DEVICE,
        )

        src = edge_index[0]
        dst = edge_index[1]

        edge_embedding = self.edge_emb(edge_type)

        for message_layer, self_layer in zip(
            self.message_layers,
            self.self_layers,
        ):
            msg = message_layer(
                torch.cat([
                    h[src],
                    edge_embedding,
                ], dim=-1)
            )

            agg = torch.zeros(
                (h.shape[0], h.shape[1]),
                dtype=h.dtype,
                device=DEVICE,
            )

            agg.index_add_(0, dst, msg)

            counts = torch.zeros(
                (h.shape[0], 1),
                dtype=h.dtype,
                device=DEVICE,
            )

            counts.index_add_(
                0,
                dst,
                torch.ones(
                    (dst.numel(), 1),
                    dtype=h.dtype,
                    device=DEVICE,
                ),
            )

            agg = agg / counts.clamp_min(1.0)

            h = F.relu(
                self_layer(h) + agg
            )

            h = F.dropout(
                h,
                p=self.dropout,
                training=self.training,
            )

        return h


class GNNOperationRanker(BaseOperationRanker):

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__(
            n_skill=n_skill,
            n_fix=n_fix,
            n_part=n_part,
            emb_dim=emb_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.graph_encoder = GraphEncoder(
            n_part=n_part,
            n_fix=n_fix,
            part_emb_dim=emb_dim,
            graph_dim=graph_dim,
            dropout=dropout,
        )

        # src embedding, dst embedding, absolute difference
        self.structural_projection = nn.Sequential(
            nn.Linear(graph_dim * 3, hidden_dim),
            nn.ReLU(),
        )

        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
        )

    def encode_operations(self, pack):
        base = self.base_operation_encoding(pack)
        node_h = self.graph_encoder(pack)

        op_nodes = torch.as_tensor(
            pack["op_graph_nodes"],
            dtype=torch.long,
            device=DEVICE,
        )

        n_ops = op_nodes.shape[0]

        a = torch.zeros(
            (n_ops, node_h.shape[1]),
            device=DEVICE,
        )

        b = torch.zeros_like(a)

        fixation_mask = op_nodes[:, 0] >= 0

        if fixation_mask.any():
            a[fixation_mask] = node_h[
                op_nodes[fixation_mask, 0]
            ]

            b[fixation_mask] = node_h[
                op_nodes[fixation_mask, 1]
            ]

        part_mask = op_nodes[:, 2] >= 0

        if part_mask.any():
            a[part_mask] = node_h[
                op_nodes[part_mask, 2]
            ]

        structural = self.structural_projection(
            torch.cat([
                a,
                b,
                torch.abs(a - b),
            ], dim=-1)
        )

        return self.combine(
            torch.cat([
                base,
                structural,
            ], dim=-1)
        )

In [ ]:
# 9. Reproducibility + pairwise-ranking training.

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_training_pairs(n_ops, k, rng):
    """
    State t:
      current = operation t
      positive = reference operation t+1
      negatives = sampled from later remaining operations t+2...
    """

    current = []
    positive = []
    negative = []
    progress = []

    for t in range(n_ops - 2):
        pool = np.arange(
            t + 2,
            n_ops,
            dtype=np.int64,
        )

        neg = rng.choice(
            pool,
            size=k,
            replace=(len(pool) < k),
        )

        current.append(t)
        positive.append(t + 1)
        negative.append(neg)
        progress.append(
            t / max(1, n_ops - 1)
        )

    return (
        np.asarray(current, dtype=np.int64),
        np.asarray(positive, dtype=np.int64),
        np.stack(negative),
        np.asarray(progress, dtype=np.float32),
    )


def build_model(model_name, vocab):
    n_skill = len(vocab["skill"])
    n_fix = len(vocab["fix"])
    n_part = len(vocab["part"])

    if model_name == "MLP":
        model = BaseOperationRanker(
            n_skill=n_skill,
            n_fix=n_fix,
            n_part=n_part,
        )

    elif model_name == "GNN":
        model = GNNOperationRanker(
            n_skill=n_skill,
            n_fix=n_fix,
            n_part=n_part,
        )

    else:
        raise ValueError(model_name)

    return model.to(DEVICE)


def train_one_model(
    model_name,
    train_batteries,
    test_battery,
    seed,
):
    set_seed(seed)

    vocab = build_fold_vocab(train_batteries)

    packs = {
        b: build_battery_pack(b, vocab)
        for b in train_batteries + [test_battery]
    }

    model = build_model(
        model_name,
        vocab,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    rng = np.random.default_rng(seed)

    for epoch in range(EPOCHS):
        order = list(train_batteries)
        rng.shuffle(order)

        model.train()

        for b in order:
            pack = packs[b]

            cur, pos, neg, prog = make_training_pairs(
                pack["n_ops"],
                NEGATIVES_PER_STATE,
                rng,
            )

            operation_repr = model.encode_operations(pack)

            cur_t = torch.as_tensor(
                cur,
                dtype=torch.long,
                device=DEVICE,
            )

            pos_t = torch.as_tensor(
                pos,
                dtype=torch.long,
                device=DEVICE,
            )

            neg_t = torch.as_tensor(
                neg,
                dtype=torch.long,
                device=DEVICE,
            )

            prog_t = torch.as_tensor(
                prog,
                dtype=torch.float32,
                device=DEVICE,
            ).unsqueeze(-1)

            current_repr = operation_repr[cur_t]
            positive_repr = operation_repr[pos_t]

            positive_score = model.score_pairs(
                current_repr,
                positive_repr,
                prog_t,
            )

            k = neg_t.shape[1]

            current_repeated = (
                current_repr[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, current_repr.shape[1])
            )

            negative_repr = operation_repr[
                neg_t.reshape(-1)
            ]

            progress_repeated = (
                prog_t[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, 1)
            )

            negative_score = model.score_pairs(
                current_repeated,
                negative_repr,
                progress_repeated,
            ).view(len(cur), k)

            loss = F.softplus(
                -(
                    positive_score[:, None]
                    - negative_score
                )
            ).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model, packs[test_battery], vocab

In [ ]:
# 10. Tie-aware ranking metrics.

def expected_tie_metrics(scores, target_position=0, tol=1e-8):
    """
    Scores correspond to candidate operations.

    Returns the expected Hits@1, Hits@3, and reciprocal rank
    under random ordering within exact/near score ties.
    """

    scores = np.asarray(scores)
    target_score = scores[target_position]

    greater = int(
        np.sum(scores > target_score + tol)
    )

    equal = int(
        np.sum(
            np.abs(scores - target_score) <= tol
        )
    )

    equal = max(equal, 1)

    hits1 = (
        max(
            0,
            min(equal, 1 - greater),
        )
        / equal
        if greater < 1
        else 0.0
    )

    hits3 = (
        max(
            0,
            min(equal, 3 - greater),
        )
        / equal
        if greater < 3
        else 0.0
    )

    ranks = np.arange(
        greater + 1,
        greater + equal + 1,
        dtype=float,
    )

    mrr = float(
        np.mean(1.0 / ranks)
    )

    return hits1, hits3, mrr

In [ ]:
# 11. Evaluate one neural model on an unseen battery.

def training_signature_set(train_batteries):
    s = set()

    for b in train_batteries:
        s.update(raw_signatures[b])

    return s


def evaluate_neural_model(
    model,
    test_pack,
    train_batteries,
):
    model.eval()

    seen_signatures = training_signature_set(
        train_batteries
    )

    rows = []

    with torch.no_grad():
        operation_repr = model.encode_operations(
            test_pack
        )

        n_ops = test_pack["n_ops"]

        for t in range(n_ops - 1):
            candidate_indices = np.arange(
                t + 1,
                n_ops,
                dtype=np.int64,
            )

            candidate_t = torch.as_tensor(
                candidate_indices,
                dtype=torch.long,
                device=DEVICE,
            )

            current_repr = (
                operation_repr[t]
                .unsqueeze(0)
                .expand(
                    len(candidate_indices),
                    -1,
                )
            )

            candidate_repr = operation_repr[
                candidate_t
            ]

            progress = torch.full(
                (len(candidate_indices), 1),
                t / max(1, n_ops - 1),
                dtype=torch.float32,
                device=DEVICE,
            )

            scores = model.score_pairs(
                current_repr,
                candidate_repr,
                progress,
            ).cpu().numpy()

            # candidate_indices[0] is the reference next operation.
            h1, h3, mrr = expected_tie_metrics(
                scores,
                target_position=0,
            )

            target_signature = (
                test_pack["full_signatures"][t + 1]
            )

            rows.append({
                "position": t,
                "hits1": h1,
                "hits3": h3,
                "mrr": mrr,
                "signature_seen_in_training":
                    target_signature in seen_signatures,
            })

    detail = pd.DataFrame(rows)

    metrics = {
        "hits1": float(detail["hits1"].mean()),
        "hits3": float(detail["hits3"].mean()),
        "mrr": float(detail["mrr"].mean()),
        "seen_signature_rate": float(
            detail["signature_seen_in_training"].mean()
        ),
    }

    seen = detail[
        detail["signature_seen_in_training"]
    ]

    unseen = detail[
        ~detail["signature_seen_in_training"]
    ]

    for label, subset in [
        ("seen", seen),
        ("unseen", unseen),
    ]:
        if len(subset) > 0:
            metrics[f"{label}_hits1"] = float(
                subset["hits1"].mean()
            )

            metrics[f"{label}_hits3"] = float(
                subset["hits3"].mean()
            )

            metrics[f"{label}_mrr"] = float(
                subset["mrr"].mean()
            )
        else:
            metrics[f"{label}_hits1"] = np.nan
            metrics[f"{label}_hits3"] = np.nan
            metrics[f"{label}_mrr"] = np.nan

    return metrics, detail

In [ ]:
# 12. Exact-operation Markov ranking baseline on the same candidate sets.
#
# This is different from the earlier signature-classification baseline.
# Candidate operations with identical signatures receive identical scores
# and are handled by the same expected tie rule.

def evaluate_markov_candidate_rank(
    train_batteries,
    test_battery,
):
    one = defaultdict(Counter)
    two = defaultdict(Counter)
    global_counts = Counter()

    for b in train_batteries:
        sig = raw_signatures[b]

        for t in range(len(sig) - 1):
            prev_sig = (
                sig[t - 1]
                if t > 0
                else "<START>"
            )

            cur_sig = sig[t]
            nxt_sig = sig[t + 1]

            one[cur_sig][nxt_sig] += 1
            two[(prev_sig, cur_sig)][nxt_sig] += 1
            global_counts[nxt_sig] += 1

    test_sig = raw_signatures[test_battery]
    train_signature_set = set(global_counts)

    rows = []

    for t in range(len(test_sig) - 1):
        prev_sig = (
            test_sig[t - 1]
            if t > 0
            else "<START>"
        )

        cur_sig = test_sig[t]

        candidate_sigs = test_sig[t + 1:]

        context2 = two.get(
            (prev_sig, cur_sig),
            Counter(),
        )

        context1 = one.get(
            cur_sig,
            Counter(),
        )

        use_two = (
            sum(context2.values()) >= 2
        )

        scores = []

        for cand_sig in candidate_sigs:
            if use_two:
                score = float(
                    context2[cand_sig]
                )
            elif sum(context1.values()) > 0:
                score = float(
                    context1[cand_sig]
                )
            else:
                score = float(
                    global_counts[cand_sig]
                )

            scores.append(score)

        h1, h3, mrr = expected_tie_metrics(
            scores,
            target_position=0,
        )

        rows.append({
            "position": t,
            "hits1": h1,
            "hits3": h3,
            "mrr": mrr,
            "signature_seen_in_training":
                test_sig[t + 1]
                in train_signature_set,
        })

    detail = pd.DataFrame(rows)

    return {
        "hits1": float(detail["hits1"].mean()),
        "hits3": float(detail["hits3"].mean()),
        "mrr": float(detail["mrr"].mean()),
        "seen_signature_rate": float(
            detail["signature_seen_in_training"].mean()
        ),
    }

In [ ]:
# 13. Smoke test one fold before the full experiment.

TEST_BATTERY = 1

train_batteries = [
    b
    for b in sorted(chains)
    if b != TEST_BATTERY
]

vocab = build_fold_vocab(train_batteries)
test_pack = build_battery_pack(
    TEST_BATTERY,
    vocab,
)

print(
    "Held-out:",
    battery_names[TEST_BATTERY],
)
print(
    "Test operations:",
    test_pack["n_ops"],
)
print(
    "Training skill vocabulary:",
    len(vocab["skill"]),
)
print(
    "Training part-class vocabulary:",
    len(vocab["part"]),
)
print(
    "Training fixation vocabulary:",
    len(vocab["fix"]),
)

markov_smoke = evaluate_markov_candidate_rank(
    train_batteries,
    TEST_BATTERY,
)

print("Markov smoke:", markov_smoke)

# One short neural fit only to catch code/runtime problems.
old_epochs = EPOCHS
EPOCHS = 2

smoke_model, smoke_pack, _ = train_one_model(
    "GNN",
    train_batteries,
    TEST_BATTERY,
    seed=SEEDS[0],
)

smoke_metrics, _ = evaluate_neural_model(
    smoke_model,
    smoke_pack,
    train_batteries,
)

EPOCHS = old_epochs

print("2-epoch GNN smoke:", smoke_metrics)
print("Smoke test complete.")

In [ ]:
# 14. Full 10-fold x 3-seed experiment.

rows = []
start_time = time.time()

battery_ids = sorted(chains)

for fold_idx, test_battery in enumerate(
    battery_ids,
    start=1,
):
    train_batteries = [
        b
        for b in battery_ids
        if b != test_battery
    ]

    print(
        f"\nFold {fold_idx}/10 — held out: "
        f"{battery_names[test_battery]}"
    )

    # Exact-operation Markov ranking baseline.
    markov_metrics = evaluate_markov_candidate_rank(
        train_batteries,
        test_battery,
    )

    rows.append({
        "battery_id": test_battery,
        "battery": battery_names[test_battery],
        "model": "Markov",
        "seed": -1,
        **markov_metrics,
    })

    for model_name in ["MLP", "GNN"]:
        for seed in SEEDS:
            model, test_pack, vocab = train_one_model(
                model_name,
                train_batteries,
                test_battery,
                seed,
            )

            metrics, detail = evaluate_neural_model(
                model,
                test_pack,
                train_batteries,
            )

            rows.append({
                "battery_id": test_battery,
                "battery": battery_names[test_battery],
                "model": model_name,
                "seed": seed,
                **metrics,
            })

            print(
                f"  {model_name} seed={seed}: "
                f"H@1={metrics['hits1']:.3f}, "
                f"H@3={metrics['hits3']:.3f}, "
                f"MRR={metrics['mrr']:.3f}"
            )

results = pd.DataFrame(rows)

elapsed = time.time() - start_time

print(
    f"\nCompleted in {elapsed/60:.1f} minutes."
)

display(results.head())

In [ ]:
# 15. Aggregate neural results across seeds.

neural = results[
    results["model"].isin(["MLP", "GNN"])
].copy()

fold_mean = (
    neural
    .groupby(
        ["battery_id", "battery", "model"],
        as_index=False,
    )
    .agg(
        hits1=("hits1", "mean"),
        hits3=("hits3", "mean"),
        mrr=("mrr", "mean"),
        seen_signature_rate=("seen_signature_rate", "mean"),
        seen_hits1=("seen_hits1", "mean"),
        unseen_hits1=("unseen_hits1", "mean"),
        seen_mrr=("seen_mrr", "mean"),
        unseen_mrr=("unseen_mrr", "mean"),
    )
)

display(
    fold_mean.sort_values(
        ["battery_id", "model"]
    )
)

summary = (
    fold_mean
    .groupby("model", as_index=False)
    .agg(
        mean_hits1=("hits1", "mean"),
        mean_hits3=("hits3", "mean"),
        mean_mrr=("mrr", "mean"),
        mean_seen_hits1=("seen_hits1", "mean"),
        mean_unseen_hits1=("unseen_hits1", "mean"),
        mean_seen_mrr=("seen_mrr", "mean"),
        mean_unseen_mrr=("unseen_mrr", "mean"),
    )
)

markov_summary = pd.DataFrame([{
    "model": "Markov",
    "mean_hits1": results.loc[
        results["model"] == "Markov",
        "hits1",
    ].mean(),
    "mean_hits3": results.loc[
        results["model"] == "Markov",
        "hits3",
    ].mean(),
    "mean_mrr": results.loc[
        results["model"] == "Markov",
        "mrr",
    ].mean(),
}])

display(summary)
display(markov_summary)

In [ ]:
# 16. Frozen continuation gate.

pivot_h1 = fold_mean.pivot(
    index="battery",
    columns="model",
    values="hits1",
)

pivot_mrr = fold_mean.pivot(
    index="battery",
    columns="model",
    values="mrr",
)

mean_mlp_h1 = float(
    pivot_h1["MLP"].mean()
)

mean_gnn_h1 = float(
    pivot_h1["GNN"].mean()
)

delta_h1 = (
    mean_gnn_h1
    - mean_mlp_h1
)

gnn_fold_wins = int(
    (
        pivot_h1["GNN"]
        > pivot_h1["MLP"]
    ).sum()
)

mean_mlp_mrr = float(
    pivot_mrr["MLP"].mean()
)

mean_gnn_mrr = float(
    pivot_mrr["GNN"].mean()
)

gate_1 = delta_h1 >= 0.05
gate_2 = gnn_fold_wins >= 7
gate_3 = mean_gnn_mrr > mean_mlp_mrr

print("=== MLP-vs-GNN LOPO DECISION SUMMARY ===")
print()
print(
    f"G1 mean Hits@1 improvement >= 5 pp: "
    f"{100*delta_h1:.2f} pp -> {gate_1}"
)
print(
    f"G2 GNN wins on >= 7/10 batteries: "
    f"{gnn_fold_wins}/10 -> {gate_2}"
)
print(
    f"G3 mean MRR improves: "
    f"MLP={mean_mlp_mrr:.4f}, "
    f"GNN={mean_gnn_mrr:.4f} -> {gate_3}"
)

continue_project = bool(
    gate_1
    and gate_2
    and gate_3
)

print()
print(
    "DECISION:",
    (
        "CONTINUE TO A STRONGER GRAPH MODEL"
        if continue_project
        else "STOP / REASSESS UNDER THE FROZEN RULE"
    ),
)

In [ ]:
# 17. Per-battery paired comparison.

comparison = (
    pivot_h1
    .assign(
        gnn_minus_mlp=lambda x:
            x["GNN"] - x["MLP"]
    )
    .sort_values(
        "gnn_minus_mlp",
        ascending=False,
    )
)

display(comparison)

In [ ]:
# 18. Save outputs.

OUT = Path(
    "/content/battery_disassembly_mlp_vs_gnn_lopo"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)

results.to_csv(
    OUT / "mlp_gnn_lopo_seed_results.csv",
    index=False,
)

fold_mean.to_csv(
    OUT / "mlp_gnn_lopo_fold_mean.csv",
    index=False,
)

summary.to_csv(
    OUT / "mlp_gnn_lopo_summary.csv",
    index=False,
)

comparison.reset_index().to_csv(
    OUT / "mlp_gnn_lopo_comparison.csv",
    index=False,
)

gate = {
    "mean_mlp_hits1": mean_mlp_h1,
    "mean_gnn_hits1": mean_gnn_h1,
    "gnn_minus_mlp_hits1": delta_h1,
    "gnn_fold_wins": gnn_fold_wins,
    "mean_mlp_mrr": mean_mlp_mrr,
    "mean_gnn_mrr": mean_gnn_mrr,
    "gate_1": gate_1,
    "gate_2": gate_2,
    "gate_3": gate_3,
    "continue_project": continue_project,
}

with open(
    OUT / "mlp_gnn_lopo_gate.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(gate, f, indent=2)

print("Saved to:", OUT)

for p in sorted(OUT.iterdir()):
    print(" -", p.name)

## What to send back

Please send the complete block beginning:

`=== MLP-vs-GNN LOPO DECISION SUMMARY ===`

and preferably upload:

- `mlp_gnn_lopo_fold_mean.csv`
- `mlp_gnn_lopo_seed_results.csv`
- `mlp_gnn_lopo_summary.csv`

If the gate passes, we do **not** immediately build a large architecture. The next step will be to inspect where graph structure helps, especially on unseen relational signatures, and then choose the smallest method extension justified by those failures.